In [1]:
from pathlib import Path
import torch
import sys
sys.path.append(str(Path.cwd().parent))
from modules.data_loading.mpd.reader import MPDConfig
from modules.data_loading.mpd.make_datasets import make_mpd_loaders
from modules.models.decode_only_transformer import ModelConfig, DecodeOnlyTransformer
import torch.nn.functional as F


In [2]:
config = MPDConfig(
    data_dir=Path.cwd().parent / 'datasets' / 'MPD' / 'data',
    min_track_freq=1,
    min_playlist_len=2,
    max_seq_len=64,
    seed=10,
    max_train_playlists=128
)


In [3]:
train_loader, val_loader, test_loader, vocab = make_mpd_loaders(
    config=config,
    batch_size=8,
    num_workers=0
)

print('Vocab size:', len(vocab))
print('Train batches:', len(train_loader))
print('Val batches:', len(val_loader))
print('Test batches:', len(test_loader))


Train playlists collected: 128
Val playlists collected: 12
Test playlists collected: 12


  0%|          | 0/128 [00:00<?, ?it/s]

Vocab size: 6645


  0%|          | 0/128 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

Train sequences kept: 128
Val sequences kept: 12
Test sequences kept: 12
Vocab size: 6645
Train batches: 16
Val batches: 2
Test batches: 2


In [4]:
batch = next(iter(train_loader))

for k, v in batch.items():
    print(k, v.shape)
    

input_ids torch.Size([8, 63])
labels torch.Size([8, 63])
attention_mask torch.Size([8, 63])


In [5]:
print(batch['input_ids'][1])
print(batch['labels'][1])
print(batch['attention_mask'][1])


tensor([   1, 1391, 3956, 3569, 2370, 4412, 5527, 6555,  464, 4778, 3705, 3122,
        4039,  321, 3229, 1009, 5201, 6325, 2364, 5870, 1245, 6383,  454, 2870,
        6583, 5991, 4854, 1690, 2226, 2446, 1434, 5356, 4325, 2176, 3753, 2656,
        2279, 1717, 4666, 3742, 1151, 2296, 1269,  278, 3966, 1161,  978, 5394,
           5, 3155,   41, 5477,  538, 2893,    0,    0,    0,    0,    0,    0,
           0,    0,    0])
tensor([1391, 3956, 3569, 2370, 4412, 5527, 6555,  464, 4778, 3705, 3122, 4039,
         321, 3229, 1009, 5201, 6325, 2364, 5870, 1245, 6383,  454, 2870, 6583,
        5991, 4854, 1690, 2226, 2446, 1434, 5356, 4325, 2176, 3753, 2656, 2279,
        1717, 4666, 3742, 1151, 2296, 1269,  278, 3966, 1161,  978, 5394,    5,
        3155,   41, 5477,  538, 2893,    2, -100, -100, -100, -100, -100, -100,
        -100, -100, -100])
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

In [6]:
seq = batch['input_ids'][1]

tokens = [vocab.decode_token(int(i)) for i in seq if i != vocab.pad_idx]

tokens[:20]


['[BOS]',
 'spotify:track:1hGy2eLcmC8eKx7qr1tOqx',
 'spotify:track:4fINc8dnfcz7AdhFYVA4i7',
 'spotify:track:4E5P1XyAFtrjpiIxkydly4',
 'spotify:track:2oENJa1T33GJ0w8dC167G4',
 'spotify:track:5BoIP8Eha5hwmRVURkC2Us',
 'spotify:track:6UaRii9AH6Zss9xNMEQ2M9',
 'spotify:track:7uKcScNXuO3MWw6LowBjW1',
 'spotify:track:0XUfyU2QviPAs6bxSpXYG4',
 'spotify:track:5dNfHmqgr128gMY2tc5CeJ',
 'spotify:track:4Ow5x7P5NAAR1jPoskudoA',
 'spotify:track:3iL2l5gUqyPS6vDwJFgJTR',
 'spotify:track:4lLtanYk6tkMvooU0tWzG8',
 'spotify:track:0O45fw2L5vsWpdsOdXwNAR',
 'spotify:track:3pfXxHoydFRfD7IBGJTQAN',
 'spotify:track:1CdkNxTlkUWR4ZnXcKES3b',
 'spotify:track:67T6l4q3zVjC5nZZPXByU8',
 'spotify:track:7d1CFwrBmH34gmS0Hkbfbt',
 'spotify:track:2nbClS09zsIAqNkshg6jnp',
 'spotify:track:6rbeWjEavBHvX2kr6lSogS']

In [7]:
model_config = ModelConfig(
    num_tracks=len(vocab),
    d_model=128,
    n_heads=4,
    n_layers=2,
    d_ff=256,
    dropout=0.0,
    max_seq_len=config.max_seq_len,
    pad_idx=vocab.pad_idx,
)

model = DecodeOnlyTransformer(model_config)

batch = next(iter(train_loader))
logits = model(batch['input_ids'], batch['attention_mask'])

print('input_ids:', batch['input_ids'].shape)
print('labels:', batch['labels'].shape)
print('logits:', logits.shape)


input_ids: torch.Size([8, 63])
labels: torch.Size([8, 63])
logits: torch.Size([8, 63, 6645])


In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    total_tokens = 0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        optimizer.zero_grad()

        logits = model(input_ids, attention_mask)

        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            labels.view(-1),
            ignore_index=-100,
        )

        loss.backward()
        optimizer.step()

        with torch.no_grad():
            valid_tokens = (labels != -100).sum().item()
            total_loss += loss.item() * valid_tokens
            total_tokens += valid_tokens

    avg_loss = total_loss / max(total_tokens, 1)
    print(f'Epoch {epoch + 1:02d} | train token CE: {avg_loss:.4f}')
    

Epoch 01 | train token CE: 82.4529
Epoch 02 | train token CE: 59.5339
Epoch 03 | train token CE: 37.2739
Epoch 04 | train token CE: 31.2495
Epoch 05 | train token CE: 28.4442
Epoch 06 | train token CE: 26.4932
Epoch 07 | train token CE: 25.0662
Epoch 08 | train token CE: 23.8480
Epoch 09 | train token CE: 22.8036
Epoch 10 | train token CE: 21.8670
Epoch 11 | train token CE: 21.0824
Epoch 12 | train token CE: 20.3464
Epoch 13 | train token CE: 19.6844
Epoch 14 | train token CE: 19.0759
Epoch 15 | train token CE: 18.5308
Epoch 16 | train token CE: 18.0159
Epoch 17 | train token CE: 17.5353
Epoch 18 | train token CE: 17.1056
Epoch 19 | train token CE: 16.6955
Epoch 20 | train token CE: 16.3152


In [9]:
model.eval()

batch = next(iter(train_loader))
input_ids = batch['input_ids'][:1].to(device)
attention_mask = batch['attention_mask'][:1].to(device)
labels = batch['labels'][:1].to(device)

with torch.no_grad():
    logits = model(input_ids, attention_mask)
    preds = logits.argmax(dim=-1)

inp = input_ids[0].tolist()
lab = labels[0].tolist()
prd = preds[0].tolist()

for i in range(min(15, len(inp))):
    in_tok = vocab.decode_token(inp[i])
    y_tok = vocab.decode_token(lab[i]) if lab[i] != -100 else '[IGN]'
    p_tok = vocab.decode_token(prd[i])
    print(f'{i:02d} | in: {in_tok}')
    print(f'   | y : {y_tok}')
    print(f'   | p : {p_tok}')
    

00 | in: [BOS]
   | y : spotify:track:5Zb94ke9RQDjpa4yKW2aoA
   | p : spotify:track:2WRh5NjSCLdaynh890QGLb
01 | in: spotify:track:5Zb94ke9RQDjpa4yKW2aoA
   | y : spotify:track:7DoBn08dqKS9MmtZiZChVD
   | p : spotify:track:1gEKRhYN6fN9hLm5Usw94N
02 | in: spotify:track:7DoBn08dqKS9MmtZiZChVD
   | y : spotify:track:02G8w6L5nlaK7M0e2Ytinu
   | p : [EOS]
03 | in: spotify:track:02G8w6L5nlaK7M0e2Ytinu
   | y : spotify:track:5HDlh6UhT3AMQs935wE1qr
   | p : spotify:track:02G8w6L5nlaK7M0e2Ytinu
04 | in: spotify:track:5HDlh6UhT3AMQs935wE1qr
   | y : spotify:track:03YuyrltrfOOtysn0HBvax
   | p : spotify:track:5HDlh6UhT3AMQs935wE1qr
05 | in: spotify:track:03YuyrltrfOOtysn0HBvax
   | y : spotify:track:2jIu30Vg6ox7sK0Oy2DK2Q
   | p : spotify:track:3UfCFlGruX0DRPCpD1V3pU
06 | in: spotify:track:2jIu30Vg6ox7sK0Oy2DK2Q
   | y : spotify:track:384N4lsAziKXZ1MlDFTUAU
   | p : spotify:track:0myCTCHVO6ULPX1SsFi4i6
07 | in: spotify:track:384N4lsAziKXZ1MlDFTUAU
   | y : spotify:track:7Il449tBVu9kGcHsEt3HH1
   |